# IGVF Catalog case studies for T2D, liver disease, and coronary artery disease, used in manuscript [To do: add bioRxiv link]

This notebook reproduces the three case study examples using the production IGVF Catalog API at `https://api.catalogkg.igvf.org/api`.

- GWAS stats are −log10 pvalue. LD is TopLD, European ancestry, r² ≥ 0.8. Genome coordinates are GRCh38.
- API source code: <https://github.com/IGVF-DACC/igvf-catalog> (`src/routers/datatypeRouters/`).

## 0. Setup

In [1]:
import re
import json
import requests
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

API = "https://api.catalogkg.igvf.org/api"
session = requests.Session()
_cache = {}


def get(path, limit=100, max_pages=50, **params):
    """GET an endpoint and follow pages. Returns a list of records (or a dict for summary endpoints).

    `limit` is the page size; each endpoint has its own maximum (noted where it is used).
    """
    params = {k: v for k, v in params.items() if v is not None}
    key = (path, limit, max_pages, tuple(sorted(params.items())))
    if key in _cache:
        return _cache[key]
    out, page = [], 0
    while page < max_pages:
        r = session.get(API + path, params=dict(params, page=page, limit=limit), timeout=120)
        if r.status_code >= 400:
            raise RuntimeError(f"{r.status_code} for {r.url}\n{r.text[:400]}")
        data = r.json()
        if isinstance(data, dict):
            _cache[key] = data
            return data
        out.extend(data)
        if len(data) < limit:
            break
        page += 1
    else:
        print(f"warning: stopped after {max_pages} pages for {path} {params}")
    _cache[key] = out
    return out


def df(rows):
    """List of records to a flat DataFrame (nested objects become dotted columns)."""
    return pd.json_normalize(rows) if rows else pd.DataFrame()


def col(frame, *candidates):
    """Name of the first candidate column present in the frame, or None."""
    for c in candidates:
        if c in frame.columns:
            return c
    return None


def keep(frame, *names):
    """Subset of columns that exist, in the order given."""
    return frame[[n for n in names if n in frame.columns]]


def bare(variant_id):
    """'variants/NC_...' to 'NC_...'."""
    return variant_id.split("/")[-1] if isinstance(variant_id, str) else variant_id


def in_context(frame, pattern, column="biological_context"):
    """Rows whose biological context matches a regex (case-insensitive)."""
    if frame.empty or column not in frame.columns:
        return frame
    return frame[frame[column].fillna("").str.contains(pattern, case=False, regex=True)]

In [2]:
# Connection check
get("/variants", rsid="rs11257655")

[{'organism': 'Homo sapiens',
  '_id': 'NC_000010.11:12265894:C:A',
  'chr': 'chr10',
  'pos': 12265894,
  'rsid': ['rs11257655'],
  'ref': 'C',
  'alt': 'A',
  'spdi': 'NC_000010.11:12265894:C:A',
  'hgvs': 'NC_000010.11:g.12265895C>A',
  'ca_id': 'CA1891048481',
  'qual': '.',
  'files_filesets': None,
  'annotations': {'bravo_af': None,
   'gnomad_af_total': None,
   'gnomad_af_afr': None,
   'gnomad_af_afr_female': None,
   'gnomad_af_afr_male': None,
   'gnomad_af_ami': None,
   'gnomad_af_ami_female': None,
   'gnomad_af_ami_male': None,
   'gnomad_af_amr': None,
   'gnomad_af_amr_female': None,
   'gnomad_af_amr_male': None,
   'gnomad_af_asj': None,
   'gnomad_af_asj_female': None,
   'gnomad_af_asj_male': None,
   'gnomad_af_eas': None,
   'gnomad_af_eas_female': None,
   'gnomad_af_eas_male': None,
   'gnomad_af_female': None,
   'gnomad_af_fin': None,
   'gnomad_af_fin_female': None,
   'gnomad_af_fin_male': None,
   'gnomad_af_male': None,
   'gnomad_af_nfe': None,
   'gnom

## 1. Shared building blocks

One small function per endpoint. All take a variant ID in the catalog's SPDI-style form, for example `NC_000010.11:12265894:C:T`.

| Function | Endpoint | What it returns |
| :-- | :-- | :-- |
| `variant_node` | `/variants` | the variant record, by rsID or ID |
| `gwas` | `/variants/phenotypes?method=GWAS` | GWAS associations, best −log10 p per trait |
| `cv2f`, `screens` | `/variants/phenotypes?method=cV2F` or `CRISPR screen` | functional scores; variant-level CRISPR screen results |
| `reporter` | `/variants/biosamples` | MPRA / STARR-seq allelic effects |
| `qtl_genes` | `/variants/genes` | eQTL / sQTL to genes |
| `caqtl` | `/variants/genomic-elements` | chromatin accessibility QTL |
| `tf_binding` | `/variants/proteins` | ADASTRA allele-specific binding, SEMVAR motif predictions |
| `e2g` | `/genomic-elements/genes?region=` | element-to-gene predictions for elements overlapping the variant |
| `ld` | `/variants/variant-ld` | LD proxies |

In [3]:
def variant_node(variant_id=None, rsid=None):
    rows = get("/variants", variant_id=variant_id, rsid=rsid)
    if not rows:
        raise LookupError(f"no variant for {variant_id or rsid}")
    return rows[0]


def variant_id_for(rsid, alt=None):
    """rsID to catalog variant ID.

    An rsID can map to several alternate alleles. Pass `alt` (for example "G") to pick one;
    otherwise all are printed and the first is returned, which may not be the allele you mean.
    """
    rows = get("/variants", rsid=rsid)
    ids = [bare(r.get("_id")) for r in rows]
    if alt is not None:
        match = [i for i in ids if i.split(":")[-1] == alt]
        if not match:
            raise LookupError(f"{rsid} has no alt allele {alt}; alleles are {ids}")
        return match[0]
    if len(ids) > 1:
        print(f"{rsid} maps to {ids}; using the first")
    return ids[0]


def gwas(variant_id, traits=None):
    """Best -log10 p per trait. `traits` is an optional regex on the trait name."""
    d = df(get("/variants/phenotypes", variant_id=variant_id, method="GWAS", limit=100))
    term = col(d, "phenotype_term", "name")
    if d.empty or term is None:
        return pd.DataFrame(columns=["trait", "neg_log10_pvalue"])
    out = (d.groupby(term)["neg_log10_pvalue"].max().sort_values(ascending=False)
             .rename_axis("trait").reset_index(name="neg_log10_pvalue"))
    if traits:
        out = out[out["trait"].str.contains(traits, case=False, regex=True)]
    return out.reset_index(drop=True)


def cv2f(variant_id):
    return keep(df(get("/variants/phenotypes", variant_id=variant_id, method="cV2F")), "biological_context", "score")


def screens(variant_id):
    d = df(get("/variants/phenotypes", variant_id=variant_id, method="CRISPR screen", verbose="true"))
    return keep(d, "phenotype_term", "crispr_modality", "perturbation", "biological_context",
                "effect_size", "z_score", "significant", "files_filesets")


def reporter(variant_id):
    d = df(get("/variants/biosamples", variant_id=variant_id))
    return keep(d, "method", "biological_context", "log2FC", "significant", "files_filesets")


def qtl_genes(variant_id):
    d = df(get("/variants/genes", variant_id=variant_id, verbose="true", limit=500))
    return keep(d, "gene.name", "gene.gene_name", "gene", "label", "method", "biological_context",
                "effect_size", "neg_log10_pvalue", "source")


def caqtl(variant_id):
    d = df(get("/variants/genomic-elements", variant_id=variant_id, limit=300))
    return keep(d, "method", "label", "biological_context", "log2FC", "neg_log10_pvalue", "beta", "source")


def tf_binding(variant_id, method):
    """method is 'ADASTRA' (observed allele-specific binding) or 'SEMVAR' (predicted motif change)."""
    d = df(get("/variants/proteins", variant_id=variant_id, method=method, verbose="true", limit=100, max_pages=30))
    tf = col(d, "protein_complex.name", "protein_complex.names", "protein_complex")
    if d.empty:
        return d
    d = d.rename(columns={tf: "tf"}) if tf else d
    if method == "SEMVAR" and "SEMpl_annotation" in d.columns:
        d = d[d["SEMpl_annotation"] != "no_binding"]
        if "variant_effect_score" in d.columns:
            d = d.reindex(d["variant_effect_score"].abs().sort_values(ascending=False).index)
        return keep(d, "tf", "SEMpl_annotation", "variant_effect_score").drop_duplicates().head(8)
    # ADASTRA: the allele with the larger -log10 adjusted p is the favored one
    if {"neg_log10_pvalue_adj_ref", "neg_log10_pvalue_adj_alt"} <= set(d.columns):
        d["favored"] = (d["neg_log10_pvalue_adj_alt"] > d["neg_log10_pvalue_adj_ref"]).map({True: "alt", False: "ref"})
        d["p_adj"] = 10 ** -d[["neg_log10_pvalue_adj_ref", "neg_log10_pvalue_adj_alt"]].max(axis=1)
    return keep(d, "tf", "biological_context", "favored", "p_adj").drop_duplicates()


E2G_METHODS = ["ENCODE-rE2G", "scE2G"]

def e2g(variant_id, gene=None, methods=E2G_METHODS):
    """Element-to-gene predictions for elements that overlap the variant position."""
    v = variant_node(variant_id)
    region = f"{v['chr']}:{v['pos']}-{v['pos'] + 1}"
    frames = [df(get("/genomic-elements/genes", region=region, method=m, verbose="true", limit=500)) for m in methods]
    d = pd.concat([f for f in frames if not f.empty], ignore_index=True) if any(not f.empty for f in frames) else pd.DataFrame()
    if d.empty:
        return d
    g = col(d, "gene.name", "gene.gene_name", "gene.symbol", "gene")
    d = d.rename(columns={g: "gene_name"}) if g else d
    if gene and "gene_name" in d.columns:
        d = d[d["gene_name"] == gene]
    return keep(d, "method", "gene_name", "biological_context", "score", "genomic_element.chr",
                "genomic_element.start", "genomic_element.end").sort_values("score", ascending=False)


def ld(variant_id, min_r2=0.8, ancestry="EUR"):
    """LD proxies as a frame with columns variant, r2."""
    d = df(get("/variants/variant-ld", variant_id=variant_id, r2=f"gte:{min_r2}", ancestry=ancestry, limit=500))
    other = col(d, "sequence_variant", "sequence_variant._id")
    if d.empty or other is None:
        return pd.DataFrame(columns=["variant", "r2"])
    out = pd.DataFrame({"variant": d[other].map(bare), "r2": d["r2"]})
    return out.drop_duplicates("variant").sort_values("r2", ascending=False).reset_index(drop=True)

### Variant profile (one variant, all evidence)
Collects every evidence type for a variant. E2G links are reported for one target gene, filtered to the relevant tissue with a regex.

Context regexes have to match every naming style in the catalog: cell lines ("HepG2"), cell ontology names ("hepatocyte") and tissue samples ("right lobe of liver from ENCDO...").

In [4]:
def variant_profile(variant_id, gene, context):
    links = e2g(variant_id, gene)
    return {
        "variant": variant_id,
        "rsid": variant_node(variant_id).get("rsid"),
        "gwas": gwas(variant_id).head(10),
        "reporter": reporter(variant_id),
        "screens": screens(variant_id),
        "cv2f": cv2f(variant_id),
        "qtl_genes": qtl_genes(variant_id),
        "caqtl": caqtl(variant_id),
        "adastra": tf_binding(variant_id, "ADASTRA"),
        "semvar": tf_binding(variant_id, "SEMVAR"),
        "e2g_in_context": in_context(links, context),
        "e2g_top": links.head(5),
    }


def show_profile(p, sections=None):
    print(f"######## {p['variant']}  {p['rsid']}")
    for k, v in p.items():
        if isinstance(v, pd.DataFrame) and (sections is None or k in sections) and not v.empty:
            print(f"== {k} ==")
            display(v)

### LD block (lead plus proxies, evidence in one context)
For the lead and each proxy: top GWAS traits, plus the evidence measured in the chosen context. Allele-specific binding is listed for all contexts and has to be judged by hand.

This makes about ten API calls per variant, so a 58-variant block takes a few minutes the first time.

In [5]:
def ld_block(lead, gene, context, min_r2=0.8, ancestry="EUR"):
    block = pd.concat([pd.DataFrame({"variant": [lead], "r2": [1.0]}), ld(lead, min_r2, ancestry)], ignore_index=True)
    rows = []
    for vid, r2 in zip(block["variant"], block["r2"]):
        try:
            g = gwas(vid).head(3)
            s = in_context(screens(vid), context)
            rp = in_context(reporter(vid), context)
            cv = in_context(cv2f(vid), context)
            links = in_context(e2g(vid, gene), context)
            ad = tf_binding(vid, "ADASTRA")
            rows.append({
                "variant": vid,
                "rsid": variant_node(vid).get("rsid"),
                "r2": r2,
                "gwas": "; ".join(f"{t} {p:.1f}" for t, p in zip(g["trait"], g["neg_log10_pvalue"])),
                "screen_hits": s[s["significant"] == True].to_dict("records") if "significant" in s.columns else [],
                "reporter_hits": rp[rp["significant"] == True].to_dict("records") if "significant" in rp.columns else [],
                "cv2f_max": cv["score"].max() if "score" in cv.columns and len(cv) else None,
                "e2g_n_contexts": len(links),
                "e2g_max_score": links["score"].max() if "score" in links.columns and len(links) else None,
                "caqtl": sorted(set(in_context(caqtl(vid), context).get("biological_context", []))),
                "adastra": sorted({f"{t} / {c}" for t, c in zip(ad.get("tf", []), ad.get("biological_context", []))}),
            })
        except Exception as e:
            rows.append({"variant": vid, "r2": r2, "error": str(e)[:200]})
    return pd.DataFrame(rows).sort_values("r2", ascending=False).reset_index(drop=True)

### Datasets measured in a tissue
`/files-filesets` is queried once per method and filtered on the sample summary in pandas. Used to choose the anchor dataset for each disease.

In [6]:
METHODS = ["CRISPR screen", "Perturb-seq", "MPRA", "STARR-seq", "caQTL", "cV2F"]

def datasets(samples, methods=METHODS):
    frames = []
    for m in methods:
        try:
            frames.append(df(get("/files-filesets", method=m, limit=500)))
        except RuntimeError as e:      # a method name the endpoint does not accept
            print(f"skipped {m}: {str(e)[:120]}")
    d = pd.concat([f for f in frames if not f.empty], ignore_index=True)
    summ = d["simple_sample_summaries"].map(lambda x: "; ".join(x) if isinstance(x, list) else str(x))
    d = d[summ.str.contains(samples, case=False, regex=True)]
    return keep(d, "_id", "method", "lab", "preferred_assay_titles", "crispr_modality", "perturbation",
                "simple_sample_summaries", "publication").sort_values("method").reset_index(drop=True)

datasets("hepg2|liver|hepat|telohaec|endothel|hcasmc|smooth muscle|pancrea|islet")

,_id,method,lab,preferred_assay_titles,crispr_modality,simple_sample_summaries
0,IGVFFI0629LUWT,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
1,IGVFFI9726GFTC,CRISPR screen,richard-sherwood,[CRISPR FACS screen],interference,[human HepG2 cell line from donor(s) IGVFDO2837BRET]
2,IGVFFI9298BISV,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
3,IGVFFI8719HCYX,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
4,IGVFFI7368COWA,CRISPR screen,danwei-huangfu,[CRISPR FACS screen],interference,[human H1-hESC differentiated cell specimen induced to progenitor cell of endocrine pancreas at ...
5,IGVFFI9134FSCT,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
6,IGVFFI5749FOUE,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
7,IGVFFI4404SMPQ,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
8,IGVFFI1678CDBR,CRISPR screen,richard-sherwood,[CRISPR FACS screen],base editing,[human HepG2 cell line from donor(s) IGVFDO2837BRET]
9,IGVFFI1068KEUM,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]


## 2. Type 2 diabetes: rs11257655 at CDC123 / CAMK1D
This case study shows how the IGVF Catalog can be queried starting from a single regulatory element and narrowed down to one likely causal variant and its target gene, by layering evidence from functional genomic assays and computational prediction methods.

In [7]:
T2D_REGION = "chr10:12265775-12266275" # 500 bp element tested by ENCODE CRISPRi in K562.

# All elements overlapping the region, including the cCRE
elements = df(get("/genomic-elements", region=T2D_REGION, limit=1000))
display(keep(elements, "name", "chr", "start", "end", "type", "method", "source_annotation", "source"))
display(elements[elements["type"].str.contains("candidate cis", case=False, na=False)] if "type" in elements.columns else elements)

,name,chr,start,end,type,source_annotation,source
0,accessible_dna_element_chr10_12265280_12266271_GRCh38_ENCFF776CZZ,chr10,12265280,12266271,accessible dna elements,None,ENCODE
1,EH38E1448677,chr10,12265766,12266116,candidate cis regulatory element,dELS: distal Enhancer-like signal,ENCODE
2,CRISPR_chr10_12265697_12265997_GRCh38_IGVFFI1168JUYR,chr10,12265697,12265997,tested elements,candidate enhancer,IGVF
3,CRISPR_chr10_12265775_12266275_GRCh38_ENCFF968BZL,chr10,12265775,12266275,tested elements,candidate enhancer,ENCODE
4,enhancer_chr10_12265789_12266289_GRCh38_IGVFFI0793NWFM,chr10,12265789,12266289,accessible dna elements,enhancer,IGVF
...,...,...,...,...,...,...,...
328,GC_Mohlke:REF_NC000010.11|12265894|C|T|MohlkeHepControls_fwd_tile1-1,chr10,12265719,12265989,tested elements,None,IGVF
329,chr10_12265730_12265979_12265895_166_C_T_ref_NC000010.11:12265894:C:T_MohlkeHepControls,chr10,12265729,12265979,tested elements,None,IGVF
330,R:chr10:12265799-12265998(-)_REF_Region_22887:001,chr10,12265798,12265998,tested elements,None,IGVF
331,chr10_12265816_12266065_12265895_80_C_T_ref_NC000010.11:12265894:C:T_309validated,chr10,12265815,12266065,tested elements,None,IGVF


,chr,start,end,strand,name,source_annotation,type,source,source_url
1,chr10,12265766,12266116,None,EH38E1448677,dELS: distal Enhancer-like signal,candidate cis regulatory element,ENCODE,https://www.encodeproject.org/files/ENCFF420VPZ


the list includes cCRE EH38E1448677 `chr10:12265766-12266116`, a distal enhancer-like element.

In [8]:
# Element-to-gene edges for elements in the region, summarised per method and gene
eg = df(get("/genomic-elements/genes", region=T2D_REGION, verbose="true", limit=500, max_pages=20))
g = col(eg, "gene.name", "gene.gene_name", "gene.symbol", "gene")
(eg.groupby(["method", g]).agg(n_contexts=("biological_context", "nunique"), max_score=("score", "max"))
   .sort_values("n_contexts", ascending=False).head(10))

n_contexts  max_score
method      gene.name                       
ENCODE-rE2G CAMK1D            192   0.946986
            CCDC3              90   0.907931
            OPTN               50   0.672786
            CDC123             22   0.531052
            NUDT5              22   0.532857
scE2G       CAMK1D             20   0.998918
ENCODE-rE2G SEC61A2            11   0.358481
scE2G       CCDC3               9   0.459688
            CDC123              3   0.369508
ENCODE-rE2G BEND7               2   0.208347

In [9]:
# Variants in the region that were tested in a reporter assay
assayed = df(get("/variants/biosamples", region=T2D_REGION, verbose="true", limit=100, max_pages=20))
vcol = col(assayed, "variant._id", "variant")
assayed["variant_id"] = assayed[vcol].map(bare)
assayed = assayed[assayed["method"].isin(["MPRA", "STARR-seq", "BlueSTARR"])]
display(keep(assayed, "variant_id", "variant.rsid", "method", "biological_context", "log2FC", "significant", "files_filesets"))
display(assayed.groupby("method").agg(n_variants=("variant_id", "nunique"),
                                      n_significant=("significant", lambda s: int((s == True).sum()))))

# Lead = the significant reporter hit with the most GWAS traits
sig = assayed.loc[assayed["significant"] == True, "variant_id"].unique()
T2D_LEAD = max(sig, key=lambda v: len(gwas(v)))
print("lead:", T2D_LEAD)
gwas(T2D_LEAD).head(6)

,variant_id,variant.rsid,method,biological_context,log2FC,significant,files_filesets
0,NC_000010.11:12265825:C:T,[rs79003453],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19096, NA19116, NA19130, NA19159, NA19197",-0.342310,False,files_filesets/IGVFFI7256ZPXC
1,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19144, NA19207, NA19214, NA19222, NA19257",-0.017794,False,files_filesets/IGVFFI0144WAIH
2,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: HG02594, HG02611, HG02623, HG02635, HG02645",-0.155677,False,files_filesets/IGVFFI0297DKAZ
3,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA18486, NA18498, NA18520, NA18912, NA19209",-0.032209,False,files_filesets/IGVFFI2408UWDF
4,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA18489, NA18908, NA18934, NA19095, NA19184",0.040584,False,files_filesets/IGVFFI2997DZFR
5,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA18505, NA18909, NA18916, NA18923, NA19175",-0.016021,False,files_filesets/IGVFFI3264WHFL
6,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19093, NA19118, NA19137, NA19138, NA19210",-0.136391,False,files_filesets/IGVFFI4012FKNC
7,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19108, NA19141, NA19146, NA19204, NA19235",-0.025357,False,files_filesets/IGVFFI5688VHRS
8,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19096, NA19116, NA19130, NA19159, NA19197",-0.137198,False,files_filesets/IGVFFI7256ZPXC
9,NC_000010.11:12265827:C:T,[rs36062557],STARR-seq,"K562 with variants from 1000 Genomes donors: NA19153, NA19160, NA19172, NA19189, NA19223",-3.196330,True,files_filesets/IGVFFI7580GCBV


,n_variants,n_significant
method,,
MPRA,4,1
STARR-seq,7,1


lead: NC_000010.11:12265894:C:T


,trait,neg_log10_pvalue
0,type 2 diabetes mellitus,63.000000
1,hba1c measurement,60.481486
2,hemoglobin a1 measurement,46.045757
3,drugs used in diabetes use measurement,39.522879
4,glucose measurement,23.744727
5,electrocardiography,10.397940


IGVF MPRA tested 4 variants inside the element, and one variant changed enhancer activity: rs11257655 (NC_000010.11:12265894:C:T), where the T allele increased activity in HepG2 hepatocytes (log2FC +0.61). The same variant had no effect in WTC-11 iPSCs or in a K562 STARR-seq screen, which is consistent with an enhancer that is active in metabolic tissue. Element-to-gene predictions link the element to CAMK1D, with ENCODE-rE2G scores up to 0.88 in pancreas and liver and scE2G scores of 0.57 in pancreatic endocrine cells and 0.37–0.42 in beta cells. It also has the most significant GWAS p-value for type 2 diabetes, followed by HbA1c and glucose.


In [10]:
T2D_CONTEXT = "pancrea|islet|endocrine|beta|liver|hepat" # show E2G predictions in speciifc cell types
t2d = variant_profile(T2D_LEAD, "CAMK1D", T2D_CONTEXT)
show_profile(t2d)

######## NC_000010.11:12265894:C:T  ['rs11257655']
== gwas ==


,trait,neg_log10_pvalue
0,type 2 diabetes mellitus,63.000000
1,hba1c measurement,60.481486
2,hemoglobin a1 measurement,46.045757
3,drugs used in diabetes use measurement,39.522879
4,glucose measurement,23.744727
5,electrocardiography,10.397940
6,diabetes mellitus,8.608169
7,body mass index,8.154902


== reporter ==


,method,biological_context,log2FC,significant,files_filesets
0,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19144, NA19207, NA19214, NA19222, NA19257",0.019645,False,files_filesets/IGVFFI0144WAIH
1,STARR-seq,"K562 with variants from 1000 Genomes donors: HG02594, HG02611, HG02623, HG02635, HG02645",-0.006225,False,files_filesets/IGVFFI0297DKAZ
2,STARR-seq,"K562 with variants from 1000 Genomes donors: NA18486, NA18498, NA18520, NA18912, NA19209",-0.026908,False,files_filesets/IGVFFI2408UWDF
3,STARR-seq,"K562 with variants from 1000 Genomes donors: NA18489, NA18908, NA18934, NA19095, NA19184",-0.029637,False,files_filesets/IGVFFI2997DZFR
4,STARR-seq,"K562 with variants from 1000 Genomes donors: NA18505, NA18909, NA18916, NA18923, NA19175",-0.083241,False,files_filesets/IGVFFI3264WHFL
5,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19093, NA19118, NA19137, NA19138, NA19210",0.264488,False,files_filesets/IGVFFI4012FKNC
6,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19108, NA19141, NA19146, NA19204, NA19235",-0.003706,False,files_filesets/IGVFFI5688VHRS
7,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19096, NA19116, NA19130, NA19159, NA19197",0.069592,False,files_filesets/IGVFFI7256ZPXC
8,STARR-seq,"K562 with variants from 1000 Genomes donors: HG02666, HG02771, HG02804, HG02811, HG02881",-0.006634,False,files_filesets/IGVFFI7903VFKP
9,MPRA,GM25256 (WTC-11) differentiated cell specimen induced to neuron from IGVFDO1756PPKO,-0.001700,False,files_filesets/IGVFFI7899LTDI


== cv2f ==


,biological_context,score
0,lung,0.923939
1,None,0.873565
2,heart,0.849560
3,HepG2,0.894413
4,K562,0.752436
5,blood,0.807444
6,GM12878,0.799598
7,kidney,0.757825
8,liver,0.930015


... (8 outputs left out)

== e2g_in_context ==


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end
298,ENCODE-rE2G,CAMK1D,pancreas from ENCDO058AAA,0.880037,chr10,12265670,12266170
399,ENCODE-rE2G,CAMK1D,liver from ENCDO662IIE,0.877296,chr10,12265741,12266241
277,ENCODE-rE2G,CAMK1D,pancreas from ENCDO424HVB,0.760211,chr10,12265651,12266151
194,ENCODE-rE2G,CAMK1D,body of pancreas from ENCDO845WKR,0.715527,chr10,12265612,12266112
84,ENCODE-rE2G,CAMK1D,body of pancreas from ENCDO451RUA,0.713176,chr10,12265589,12266513
73,ENCODE-rE2G,CAMK1D,body of pancreas from ENCDO271OUW,0.686976,chr10,12265587,12266087
224,ENCODE-rE2G,CAMK1D,right lobe of liver from ENCDO912EIY,0.684838,chr10,12265625,12266125
203,ENCODE-rE2G,CAMK1D,pancreas from ENCDO528BHB,0.683848,chr10,12265614,12266114
152,ENCODE-rE2G,CAMK1D,body of pancreas from ENCDO793LXB,0.609704,chr10,12265601,12266101
491,scE2G,CAMK1D,pancreatic ductal cell,0.597790,chr10,12265609,12266109


== e2g_top ==


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end
496,scE2G,CAMK1D,cardiac muscle cell,0.998918,chr10,12265615,12266115
413,ENCODE-rE2G,CAMK1D,"hematopoietic multipotent progenitor cell from ENCDO937OUY treated with erythropoietin, hydrocor...",0.946986,chr10,12265747,12266247
430,ENCODE-rE2G,CAMK1D,"hematopoietic multipotent progenitor cell from ENCDO937OUY treated with erythropoietin, hydrocor...",0.942198,chr10,12265756,12266256
374,ENCODE-rE2G,CAMK1D,"common myeloid progenitor, CD34-positive from ENCDO147FTI",0.931918,chr10,12265731,12266231
323,ENCODE-rE2G,CAMK1D,"common myeloid progenitor, CD34-positive from ENCDO015OUZ",0.931312,chr10,12265711,12266211


### Evidence summary

| Evidence | Source and context | Result |
| :---- | :---- | :---- |
| Variant effect on enhancer activity | IGVF MPRA, HepG2 (Mohlke lab, IGVFFI4328UUGV) | **\+0.61, significant**; 1 of 4 variants tested in the element |
|  | IGVF MPRA, WTC-11; IGVF STARR-seq, K562 | Not significant |
| Element → gene | ENCODE-rE2G, pancreas and liver | CAMK1D, up to 0.88 (10 pancreas and 7 liver samples) |
|  | IGVF scE2G, pancreatic endocrine cell and beta cell | CAMK1D, 0.57 and 0.37 |
| Chromatin | ENCODE caQTL | Liver |
| Functional score | IGVF cV2F | Liver 0.93 |
| Disease association | GWAS | T2D 63.0; HbA1c; glucose |
| Expression | EBI eQTL Catalogue | CAMK1D, blood 33.6, positive slope |
| TF binding | IGVF SEMVAR prediction | FOXA1 / FOXA2 binding created |
|  | ADASTRA | FOXA1, alt favored: LNCaP adjusted p 8e-13, T47D 3.5e-4 |

## 3. Liver disease: GCKR and HNF1A in HepG2 LDL-C uptake screens

Starting from two IGVF variant screens for LDL-C uptake in HepG2 cells:

In [11]:
LIVER_FILES = ["IGVFFI9726GFTC", "IGVFFI1678CDBR"]

screen_rows = {}
for f in LIVER_FILES:
    d = df(get("/phenotypes/variants", files_fileset=f, verbose="true", limit=100, max_pages=60))
    vcol = col(d, "variant._id", "variant")
    d["variant_id"] = d[vcol].map(bare)
    screen_rows[f] = d
    mod = col(d, "crispr_modality", "perturbation")
    print(f, "| modality:", d[mod].dropna().unique().tolist() if mod else "?",
          "| tested:", d["variant_id"].nunique(),
          "| significant:", d.loc[d["significant"] == True, "variant_id"].nunique())

IGVFFI9726GFTC | modality: ['interference'] | tested: 2357 | significant: 240
IGVFFI1678CDBR | modality: ['base editing'] | tested: 1262 | significant: 173


CRISPRi 240 significant of 2,357 targeted variants; base editing 173 of 1,262.

In [12]:
# Significant hits that also have a liver-trait GWAS association
LIVER_TRAITS = ("fatty liver|nafld|masld|steato|liver fat|cirrhosis|alanine aminotransferase|aspartate aminotransferase|"
                "gamma-glutamyl|liver disease|hepatocellular|cholangitis|alkaline phosphatase|bilirubin")

hits = []
for f, d in screen_rows.items():
    mod = col(d, "crispr_modality", "perturbation")
    sig = d[d["significant"] == True]
    for i, (_, row) in enumerate(sig.iterrows(), 1):
        if i % 50 == 0:
            print(f, i, "/", len(sig))
        try:
            g = gwas(row["variant_id"], LIVER_TRAITS).head(4)
        except Exception as e:
            print("failed", row["variant_id"], str(e)[:100]); continue
        if len(g):
            hits.append({"variant": row["variant_id"], "rsid": row.get("variant.rsid"), "file": f,
                         "modality": row.get(mod) if mod else None, "effect": row.get("effect_size"),
                         "z": row.get("z_score"), "top_neg_log10_p": g["neg_log10_pvalue"].iloc[0],
                         "traits": "; ".join(f"{t} {p:.1f}" for t, p in zip(g["trait"], g["neg_log10_pvalue"]))})
liver_hits = pd.DataFrame(hits).sort_values("top_neg_log10_p", ascending=False).reset_index(drop=True)
liver_hits.head(40)

IGVFFI9726GFTC 50 / 240
IGVFFI9726GFTC 100 / 240
IGVFFI9726GFTC 150 / 240
IGVFFI9726GFTC 200 / 240
IGVFFI1678CDBR 50 / 173
IGVFFI1678CDBR 100 / 173
IGVFFI1678CDBR 150 / 173


,variant,rsid,file,modality,effect,z,top_neg_log10_p,traits
0,NC_000002.12:27508072:T:C,[rs1260326],IGVFFI1678CDBR,base editing,-0.241260,-1.971339,297.000000,serum gamma-glutamyl transferase measurement 297.0; alkaline phosphatase measurement 185.9; scle...
1,NC_000011.10:61824889:A:G,[rs174566],IGVFFI9726GFTC,interference,0.069919,2.417193,64.154902,alkaline phosphatase measurement 64.2
2,NC_000012.12:124827878:G:C,[rs10846744],IGVFFI9726GFTC,interference,-0.102924,-3.418373,12.698970,bilirubin measurement 12.7
3,NC_000012.12:124824135:G:A,[rs10846742],IGVFFI1678CDBR,base editing,1.152697,2.223194,10.397940,bilirubin measurement 10.4
4,NC_000019.10:11120526:G:A,[rs72658867],IGVFFI9726GFTC,interference,-0.086433,-3.299685,10.200659,bilirubin measurement 10.2
5,NC_000012.12:120978818:C:G,[rs1169289],IGVFFI9726GFTC,interference,-0.069172,-2.588353,9.000000,non-alcoholic fatty liver disease 9.0
6,NC_000011.10:61855667:C:T,[rs174601],IGVFFI9726GFTC,interference,0.078093,2.983341,8.522879,alkaline phosphatase measurement 8.5
7,NC_000002.12:27375229:T:C,[rs4665972],IGVFFI1678CDBR,base editing,-0.699801,-2.097437,8.221849,alkaline phosphatase measurement 8.2


GCKR is the strongest liver-disease locus. rs1260326 (GCKR L446P, p.Leu446Pro; known in the literature as P446L) is a base-editing hit (effect −0.24) with GGT 297, alkaline phosphatase 186, sclerosing cholangitis 22.7, NAFLD 14.5, liver fat 7.4. rs1169289 (HNF1A, NAFLD 9.0) is a CRISPRi hit (effect −0.07).

In [13]:
# Checks the canonical risk variants for regulatory evidence and coding consequences.
# rsID -> alternate allele of the known risk variant (several of these rsIDs have more than one alt allele)
MASLD_ALLELES = {
    "rs738409": "G",      # PNPLA3 I148M, C>G
    "rs58542926": "T",    # TM6SF2 E167K, C>T
    "rs2642438": "G",     # MTARC1 T165A, A>G
    "rs72613567": None,   # HSD17B13 splice insertion, single allele
    "rs641738": "C",      # MBOAT7 / TMC4, T>C
    "rs1260326": "C",     # GCKR L446P, T>C
    "rs1169289": "G",     # HNF1A, C>G
}
MASLD_TRAITS = ("fatty liver|nafld|liver fat|cirrhosis|alanine aminotransferase|aspartate aminotransferase|"
                "gamma-glutamyl|alkaline phosphatase|bilirubin")

rows = []
for rs, alt in MASLD_ALLELES.items():
    vid = variant_id_for(rs, alt)
    g = gwas(vid, MASLD_TRAITS).head(4)
    s, cv, rp = screens(vid), cv2f(vid), reporter(vid)
    try:
        coding = get("/variants/coding-variants", variant_id=vid, limit=5)
    except RuntimeError:
        coding = []
    liver_cv = cv.loc[cv["biological_context"] == "liver", "score"].max() if len(cv) else None
    rows.append({"rsid": rs, "variant": vid,
                 "gwas": "; ".join(f"{t} {p:.1f}" for t, p in zip(g["trait"], g["neg_log10_pvalue"])),
                 "screens": s.to_dict("records"), "cv2f_liver": liver_cv,
                 "reporter": rp.to_dict("records"),
                 "coding": [c.get("name") or c.get("_id") for c in coding][:1]})
pd.DataFrame(rows)

,rsid,variant,gwas,screens,cv2f_liver,reporter,coding
0,rs738409,NC_000022.11:43928846:C:G,aspartate aminotransferase measurement 323.3; serum alanine aminotransferase measurement 317.2; ...,[],NaN,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: HG0...",[PNPLA3_ENST00000216180_p.Ile148Met_c.444C>G]
1,rs58542926,NC_000019.10:19268739:C:T,alkaline phosphatase measurement 170.5; serum alanine aminotransferase measurement 125.2; liver ...,[],NaN,[],[TM6SF2_ENST00000389363_p.Glu167Lys_c.499G>A]
2,rs2642438,NC_000001.11:220796685:A:G,serum alanine aminotransferase measurement 71.5; alkaline phosphatase measurement 49.2; cirrhosi...,[],0.784787,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: NA1...",[MTARC1_ENST00000366910_p.Thr165Ala_c.493A>G]
3,rs72613567,NC_000004.12:87310240:A:AA,serum alanine aminotransferase measurement 96.0; aspartate aminotransferase measurement 37.4; as...,[],NaN,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: NA1...",[]
4,rs641738,NC_000019.10:54173067:T:C,,[],0.770347,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: NA1...",[TMC4_ENST00000617472_p.Glu17Gly_c.50A>G]
5,rs1260326,NC_000002.12:27508072:T:C,serum gamma-glutamyl transferase measurement 297.0; alkaline phosphatase measurement 185.9; non-...,"[{'phenotype_term': 'LDL-C uptake', 'crispr_modality': 'base editing', 'biological_context': 'hu...",0.855968,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: HG0...",[GCKR_ENST00000264717_p.Leu446Pro_c.1337T>C]
6,rs1169289,NC_000012.12:120978818:C:G,non-alcoholic fatty liver disease 9.0,"[{'phenotype_term': 'LDL-C uptake', 'crispr_modality': 'interference', 'biological_context': 'hu...",NaN,"[{'method': 'STARR-seq', 'biological_context': 'K562 with variants from 1000 Genomes donors: NA1...",[]


PNPLA3 I148M (rs738409; cirrhosis 290), TM6SF2 E167K (rs58542926), MTARC1 (rs2642438) and HSD17B13 (rs72613567) have strong liver GWAS and cV2F support but no regulatory-assay signal. The HSD17B13 splice variant was tested by HepG2 MPRA and was null (log2FC +0.10).

In [14]:
# overlapping with other predictions and assays in catalog
LIVER_CONTEXT = "liver|hepat|hepg2"
for vid, gene in [("NC_000002.12:27508072:T:C", "GCKR"),      # rs1260326, GCKR L446P (p.Leu446Pro)
                  ("NC_000002.12:27518369:T:C", "GCKR"),      # rs780094, proxy
                  ("NC_000012.12:120978818:C:G", "HNF1A")]:   # rs1169289
    show_profile(variant_profile(vid, gene, LIVER_CONTEXT),
                 sections=["gwas", "screens", "cv2f", "adastra", "semvar", "e2g_in_context"])

######## NC_000002.12:27508072:T:C  ['rs1260326']
== gwas ==


,trait,neg_log10_pvalue
0,triglyceride measurement,323.306215
1,sex hormone-binding globulin measurement,307.652656
2,c-reactive protein measurement,307.652656
3,serum gamma-glutamyl transferase measurement,297.000000
4,urate measurement,286.920819
5,body mass index,261.698970
6,physical activity,258.522879
7,igf-1 measurement,245.537602
8,serum albumin measurement,187.031517
9,alkaline phosphatase measurement,185.853872


== screens ==


,phenotype_term,crispr_modality,biological_context,effect_size,z_score,significant,files_filesets
0,LDL-C uptake,base editing,human HepG2 cell line from donor(s) IGVFDO2837BRET,-0.241260,-1.971339,True,files_filesets/IGVFFI1678CDBR
1,LDL-C uptake,interference,human HepG2 cell line from donor(s) IGVFDO2837BRET,0.018041,0.644428,False,files_filesets/IGVFFI9726GFTC


== cv2f ==


,biological_context,score
0,None,0.856624
1,HepG2,0.758453
2,liver,0.855968


... (18 outputs left out)

== semvar ==


,tf,SEMpl_annotation,variant_effect_score
0,TFAP4,binding_created,1.668791
1,HOXB7,binding_ablated,-0.318231
2,NFATC3,binding_ablated,-0.287159
7,ATF6,binding_created,0.120255
5,IKZF1,binding_decreased,-0.064894


== e2g_in_context ==


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end
93,ENCODE-rE2G,HNF1A,liver from ENCDO987XTQ,0.999997,chr12,120978190,120979068
30,ENCODE-rE2G,HNF1A,HepG2,0.999997,chr12,120977998,120979005
296,scE2G,HNF1A,hepatocyte,0.996167,chr12,120978275,120979021


- rs1260326: base editing significant (−0.24), CRISPRi not significant; cV2F liver 0.86.
- rs780094: ADASTRA RARA in HepG2, alt favored, adjusted p 0.022; SEMVAR FOXA1 site created; GWAS GGT 129, ALP 102, NAFLD 7.7.
- rs1169289: CRISPRi significant (−0.07); ENCODE-rE2G HepG2 / liver → HNF1A about 1.0; scE2G hepatocyte 0.996.

In [15]:
# check variants in LD blocks
gckr_ld = ld_block("NC_000002.12:27508072:T:C", "GCKR", LIVER_CONTEXT)
gckr_ld

,variant,rsid,r2,gwas,screen_hits,reporter_hits,cv2f_max,e2g_n_contexts,e2g_max_score,caqtl,adastra
0,NC_000002.12:27508072:T:C,[rs1260326],1.000,triglyceride measurement 323.3; sex hormone-binding globulin measurement 307.7; c-reactive prote...,"[{'phenotype_term': 'LDL-C uptake', 'crispr_modality': 'base editing', 'biological_context': 'hu...",[],0.855968,2,0.530837,[],[]
1,NC_000002.12:27375229:T:C,[rs4665972],0.919,triglyceride measurement 323.3; alcohol drinking 246.7; alcohol consumption measurement 237.0,"[{'phenotype_term': 'LDL-C uptake', 'crispr_modality': 'base editing', 'biological_context': 'hu...",[],NaN,0,NaN,[],[]
2,NC_000002.12:27518369:T:C,[rs780094],0.918,triglyceride measurement 323.3; c-reactive protein measurement 195.7; sex hormone-binding globul...,[],[],NaN,1,0.203931,[],[RARA / HepG2 (hepatoblastoma)]
3,NC_000002.12:27519735:T:C,[rs780093],0.912,glomerular filtration rate 61.5; total cholesterol measurement 46.5; reticulocyte measurement 19.2,"[{'phenotype_term': 'LDL-C uptake', 'crispr_modality': 'base editing', 'biological_context': 'hu...",[],NaN,0,NaN,[],[]
4,NC_000002.12:27526125:TTTTTTTTTT:TTTTTTTTT,[rs11336847],0.909,monocyte percentage of leukocytes 33.9; granulocyte percentage of myeloid white cells 27.0; alco...,[],[],NaN,0,NaN,[],[]
5,NC_000002.12:27529595:G:A,[rs11127048],0.825,vitamin d measurement 307.7; coffee consumption measurement 16.1; alcohol consumption measuremen...,[],[],NaN,0,NaN,[],[]


rs4665972 (r² 0.92, base editing −0.70), rs780093 (r² 0.91, base editing +0.52) and rs780094 (r² 0.92, not significant in the screens, RARA binding in HepG2).

In [16]:
# predicted effect of the GCKR missense variant (coding-variant scores, e.g. MutPred2)
r = session.get(API + "/coding-variants/phenotypes/score-summary",
                params={"variant_id": "NC_000002.12:27508072:T:C"}, timeout=120)
df(r.json())

,variant_id,hgvsp,gene_name,transcript_id,dataType,score,portalLink
0,variants/NC_000002.12:27508072:T:C,p.Leu446Pro,GCKR,ENST00000264717,MutPred2,0.075458,https://data.igvf.org/tabular-files/IGVFFI6893ZOAA


### Evidence summary

| Variant | Source | Results |
| :-- | :-- | :-- |
| rs1260326 | coding variant | GCKR L446P (p.Leu446Pro), missense |
| rs1260326 | MutPred2 (coding variant score) | 0.08 |
| rs1260326 | LDL-C uptake screens | base editing −0.24 |
| rs1260326 | GWAS | GGT 297; ALP 186 |
| rs1260326 | cV2F, liver | 0.86 |
| rs780094 | TopLD EUR r² with rs1260326 | 0.92 |
| rs780094 | ADASTRA | RARA in HepG2, adjusted p 0.022 |
| rs1169289 | LDL-C uptake screens | CRISPRi −0.07 |
| rs1169289 | GWAS | NAFLD 9.0 |
| rs1169289 | element to HNF1A, liver | rE2G / scE2G up to 1.0 |

### Summary
Starting from two IGVF variant screens for LDL-C uptake in HepG2 cells, the catalog picks out two liver-disease variants and separates them by mechanism. rs1260326, a missense variant in GCKR (L446P), responds to base editing but not to CRISPRi and carries the strongest liver-trait GWAS signal; its non-coding LD proxy rs780094 shows allele-specific RARA binding in HepG2. rs1169289 responds to CRISPRi and lies in an element at the HNF1A start site that is linked to HNF1A in liver. The well-known coding MASLD variants (PNPLA3, TM6SF2, MTARC1, HSD17B13) have strong GWAS support but no hit in these screens.

## 4. Coronary artery disease: 9p21 in smooth muscle and NOS3 in endothelium

In [17]:
#Vascular perturbation datasets
vascular = datasets("telohaec|endothel|hcasmc|smooth muscle|aort|coronary", methods=["CRISPR screen", "Perturb-seq"])
vascular

,_id,method,lab,preferred_assay_titles,crispr_modality,simple_sample_summaries
0,IGVFFI0629LUWT,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
1,IGVFFI1068KEUM,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
2,IGVFFI4404SMPQ,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
3,IGVFFI5749FOUE,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
4,IGVFFI6843QWSB,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
5,IGVFFI8719HCYX,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
6,IGVFFI9134FSCT,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
7,IGVFFI9298BISV,CRISPR screen,jesse-engreitz,[CRISPR FlowFISH screen],interference,[human TeloHAEC cell line from donor(s) IGVFDO3459DFTW]
8,IGVFFI0206LUDV,Perturb-seq,thomas-quertermous,[Parse Perturb-seq],interference,[human HCASMC-hTERT cell line from donor(s) IGVFDO9735BPCD]
9,IGVFFI1168JUYR,Perturb-seq,thomas-quertermous,[TAP-seq],interference,[human HCASMC-hTERT cell line from donor(s) IGVFDO9735BPCD]


8 TeloHAEC CRISPRi FlowFISH files (Engreitz lab), 4 HCASMC TAP-seq / Parse Perturb-seq files (Quertermous lab), 2 WTC-11 endothelial TAP-seq files.

In [18]:
#Significant elements in a vascular screen that contain CAD-related GWAS variants.
#The GWAS lookup is cached per element, so replicate screens that test the same elements are fast. `IGVFFI0206LUDV` (Parse Perturb-seq) is very large and is left out.

CAD_TRAITS = "coronary|myocardial|atheroscl|cardiovascular|blood pressure|hypertens"
HCASMC_FILE = "IGVFFI3434YAPX"
TELOHAEC_FILES = ["IGVFFI9134FSCT", "IGVFFI1068KEUM", "IGVFFI5749FOUE", "IGVFFI9298BISV",
                  "IGVFFI6843QWSB", "IGVFFI0629LUWT", "IGVFFI4404SMPQ", "IGVFFI8719HCYX"]


def gwas_in_region(region, traits):
    """GWAS variants inside a region, best -log10 p per variant and trait."""
    d = df(get("/variants/phenotypes", region=region, method="GWAS", verbose="true", limit=100, max_pages=30))
    term = col(d, "phenotype_term", "name")
    if d.empty or term is None:
        return pd.DataFrame()
    d = d[d[term].str.contains(traits, case=False, regex=True, na=False)]
    if d.empty:
        return d
    vcol = col(d, "variant._id", "variant")
    d = d.assign(variant_id=d[vcol].map(bare), rsid=d.get("variant.rsid"))
    out = d.groupby(["variant_id", term])["neg_log10_pvalue"].max().reset_index()
    out.columns = ["variant", "trait", "neg_log10_p"]
    return out.merge(d[["variant_id", "rsid"]].astype(str).drop_duplicates("variant_id"),
                     left_on="variant", right_on="variant_id").drop(columns="variant_id")


def screen_hits_with_gwas(file, traits):
    eg = df(get("/genomic-elements/genes", files_fileset=file, verbose="true", limit=500, max_pages=60))
    if eg.empty:
        return eg
    g = col(eg, "gene.name", "gene.gene_name", "gene.symbol", "gene")
    c, s, e = (col(eg, f"genomic_element.{x}", f"element.{x}", x) for x in ("chr", "start", "end"))
    eg = eg[eg["significant"] == True].assign(
        element=lambda x: x[c] + ":" + x[s].astype(int).astype(str) + "-" + x[e].astype(int).astype(str),
        gene_name=lambda x: x[g])
    out = []
    for element, grp in eg.groupby("element"):
        gw = gwas_in_region(element, traits)
        if gw.empty:
            continue
        for _, r in grp.iterrows():
            out.append(gw.assign(element=element, gene=r["gene_name"], log2FC=r.get("log2FC"),
                                 p_adj=r.get("p_value_adj"), file=file))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


frames = []
for f in [HCASMC_FILE] + TELOHAEC_FILES:
    try:
        d = screen_hits_with_gwas(f, CAD_TRAITS)
    except Exception as e:            # one failing file should not stop the rest
        print(f, "failed:", str(e)[:150]); continue
    print(f, len(d), "rows")
    frames.append(d)
cad_hits = pd.concat([x for x in frames if not x.empty], ignore_index=True) if frames else pd.DataFrame()
cad_hits.sort_values("neg_log10_p", ascending=False).head(30)

IGVFFI3434YAPX 5 rows
IGVFFI9134FSCT 10 rows
IGVFFI1068KEUM 1 rows
IGVFFI5749FOUE 10 rows
IGVFFI9298BISV 0 rows
IGVFFI6843QWSB 10 rows
IGVFFI0629LUWT 10 rows
IGVFFI4404SMPQ 0 rows
IGVFFI8719HCYX 0 rows


,variant,trait,neg_log10_p,rsid,element,gene,log2FC,p_adj,file
31,NC_000007.14:150993087:C:T,hypertensive disorder,62.107905,['rs3918226'],chr7:150992690-150993190,NOS3,-2.280892,8.620000e-21,IGVFFI6843QWSB
41,NC_000007.14:150993087:C:T,hypertensive disorder,62.107905,['rs3918226'],chr7:150992690-150993190,NOS3,-2.213126,3.700000e-23,IGVFFI0629LUWT
21,NC_000007.14:150993087:C:T,hypertensive disorder,62.107905,['rs3918226'],chr7:150992690-150993190,NOS3,-2.178551,4.740000e-28,IGVFFI5749FOUE
10,NC_000007.14:150993087:C:T,hypertensive disorder,62.107905,['rs3918226'],chr7:150992690-150993190,NOS3,-2.363268,3.430000e-15,IGVFFI9134FSCT
27,NC_000007.14:150993087:C:T,cardiovascular disorder,48.698970,['rs3918226'],chr7:150992690-150993190,NOS3,-2.280892,8.620000e-21,IGVFFI6843QWSB
17,NC_000007.14:150993087:C:T,cardiovascular disorder,48.698970,['rs3918226'],chr7:150992690-150993190,NOS3,-2.178551,4.740000e-28,IGVFFI5749FOUE
37,NC_000007.14:150993087:C:T,cardiovascular disorder,48.698970,['rs3918226'],chr7:150992690-150993190,NOS3,-2.213126,3.700000e-23,IGVFFI0629LUWT
6,NC_000007.14:150993087:C:T,cardiovascular disorder,48.698970,['rs3918226'],chr7:150992690-150993190,NOS3,-2.363268,3.430000e-15,IGVFFI9134FSCT
4,NC_000009.12:22103341:T:G,myocardial infarction,44.420216,['rs1537373'],chr9:22103301-22103601,CDKN2A,-0.090181,8.330000e-05,IGVFFI3434YAPX
2,NC_000009.12:22103341:T:G,myocardial infarction,44.420216,['rs1537373'],chr9:22103301-22103601,CDKN2B,-0.172535,1.140000e-05,IGVFFI3434YAPX


In [19]:
# One row per element and gene
(cad_hits.groupby(["element", "gene"])
         .agg(files=("file", "nunique"), variants=("rsid", lambda s: sorted(set(s))),
              best_gwas=("neg_log10_p", "max"), min_log2FC=("log2FC", "min"))
         .sort_values("best_gwas", ascending=False).reset_index())

,element,gene,files,variants,best_gwas,min_log2FC
0,chr7:150992690-150993190,NOS3,4,[['rs3918226']],62.107905,-2.363268
1,chr9:22103301-22103601,CDKN2A,1,[['rs1537373']],44.420216,-0.090181
2,chr9:22103301-22103601,CDKN2B,1,[['rs1537373']],44.420216,-0.172535
3,chr11:103798072-103798372,MRPL17,1,[['rs2019090']],17.621602,-0.045003
4,chr15:81085102-81085602,TLNRD1,1,[['rs1879454']],9.397940,-0.602707


from the HCASMC file, `chr9:22103301-22103601` (9p21) → CDKN2B and CDKN2A, containing rs1537373. From the TeloHAEC files, the NOS3 promoter element `chr7:150992690-150993190` (rs3918226, rs2070744) and a TLNRD1 element containing rs1879454 (CAD 9.4).

In [20]:
def element_genes(chr_, start, end, methods=("CRISPR screen", "Perturb-seq")):
    d = df(get("/genomic-elements/genes", region=f"{chr_}:{start}-{end}", verbose="true", limit=500, max_pages=20))
    if d.empty:
        return d
    s, e = col(d, "genomic_element.start", "element.start", "start"), col(d, "genomic_element.end", "element.end", "end")
    d = d[d["method"].isin(methods) & (d[s] == start) & (d[e] == end)]     # exact element only
    g = col(d, "gene.name", "gene.gene_name", "gene.symbol", "gene")
    d = d.rename(columns={g: "gene_name"})
    out = keep(d, "gene_name", "method", "biological_context", "log2FC", "p_value_adj", "neg_log10_pvalue_adj",
               "significant", "files_filesets")
    by = col(out, "p_value_adj", "neg_log10_pvalue_adj")
    return out.sort_values(by, ascending=(by == "p_value_adj")) if by else out

element_genes("chr9", 22103301, 22103601)

,gene_name,method,biological_context,log2FC,p_value_adj,neg_log10_pvalue_adj,significant,files_filesets
1,CDKN2B,Perturb-seq,HCASMC-hTERT,-0.172535,0.000011,4.943095,True,files_filesets/IGVFFI3434YAPX
2,CDKN2A,Perturb-seq,HCASMC-hTERT,-0.090181,0.000083,4.079355,True,files_filesets/IGVFFI3434YAPX
3,DMRTA1,Perturb-seq,HCASMC-hTERT,0.731797,0.621217,0.206756,False,files_filesets/IGVFFI3434YAPX
6,CDKN2B-AS1,Perturb-seq,HCASMC-hTERT,-0.096174,0.671333,0.173062,False,files_filesets/IGVFFI3434YAPX
0,MTAP,Perturb-seq,HCASMC-hTERT,-0.026789,0.978327,0.009516,False,files_filesets/IGVFFI3434YAPX
4,IFNE,Perturb-seq,HCASMC-hTERT,-0.031244,1.000000,0.000000,False,files_filesets/IGVFFI3434YAPX
5,KLHL9,Perturb-seq,HCASMC-hTERT,0.000358,1.000000,0.000000,False,files_filesets/IGVFFI3434YAPX


CDKN2B log2FC −0.17 (adjusted p 1.1e-5) and CDKN2A −0.09 (8.3e-5) significant; MTAP and CDKN2B-AS1 not significant.

In [21]:
element_genes("chr7", 150992690, 150993190)   # NOS3 promoter

,gene_name,method,biological_context,log2FC,p_value_adj,neg_log10_pvalue_adj,significant,files_filesets
1,NOS3,CRISPR screen,human TeloHAEC cell line from donor(s) IGVFDO3459DFTW,-2.178551,4.740000e-28,27.324222,True,files_filesets/IGVFFI5749FOUE
0,NOS3,CRISPR screen,human TeloHAEC cell line from donor(s) IGVFDO3459DFTW,-2.213126,3.700000e-23,22.431798,True,files_filesets/IGVFFI0629LUWT
2,NOS3,CRISPR screen,human TeloHAEC cell line from donor(s) IGVFDO3459DFTW,-2.280892,8.620000e-21,20.064493,True,files_filesets/IGVFFI6843QWSB
3,NOS3,CRISPR screen,human TeloHAEC cell line from donor(s) IGVFDO3459DFTW,-2.363268,3.430000e-15,14.464706,True,files_filesets/IGVFFI9134FSCT


NOS3 reduced in all four replicate screens, log2FC −2.2 to −2.4, adjusted p as low as 5e-28.

In [22]:
SMC_CONTEXT = "smooth muscle|artery|aort|coronary|vascul"
EC_CONTEXT = "endothel|aort|artery|coronary|huvec|vascul"

show_profile(variant_profile("NC_000009.12:22103341:T:G", "CDKN2B", SMC_CONTEXT),          # rs1537373
             sections=["gwas", "reporter", "adastra", "e2g_in_context"])
display(in_context(e2g("NC_000009.12:22103341:T:G", "CDKN2A"), SMC_CONTEXT))               # same element, CDKN2A

for rs, alt in [("rs3918226", None), ("rs2070744", "T")]:                                   # NOS3 promoter variants
    show_profile(variant_profile(variant_id_for(rs, alt), "NOS3", EC_CONTEXT),
                 sections=["gwas", "reporter", "adastra", "e2g_in_context"])

######## NC_000009.12:22103341:T:G  ['rs1537373']
== gwas ==


,trait,neg_log10_pvalue
0,myocardial infarction,44.420216
1,brain aneurysm,28.522879
2,coronary artery calcification,10.397940
3,parental longevity,9.886057


== reporter ==


,method,biological_context,log2FC,significant,files_filesets
0,STARR-seq,"K562 with variants from 1000 Genomes donors: NA18486, NA18498, NA18520, NA18912, NA19209",-0.003730,False,files_filesets/IGVFFI2408UWDF
1,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19093, NA19118, NA19137, NA19138, NA19210",-0.046784,False,files_filesets/IGVFFI4012FKNC
2,STARR-seq,"K562 with variants from 1000 Genomes donors: NA19108, NA19141, NA19146, NA19204, NA19235",0.026524,False,files_filesets/IGVFFI5688VHRS
3,STARR-seq,"K562 with variants from 1000 Genomes donors: NA18511, NA19147, NA19185, NA19200, NA19248",0.022631,False,files_filesets/IGVFFI9329ALOP
4,MPRA,GM25256 (WTC-11) differentiated cell specimen induced to neuron from IGVFDO1756PPKO,0.030300,False,files_filesets/IGVFFI7899LTDI
5,MPRA,GM25256 (WTC-11),-0.010200,False,files_filesets/IGVFFI9903YDJU
6,MPRA,HepG2,1.033895,True,files_filesets/IGVFFI4328UUGV


== e2g_in_context ==


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end
381,ENCODE-rE2G,CDKN2B,brain microvascular endothelial cell from ENCDO227AAA,0.637010,chr9,22103222,22104117
327,ENCODE-rE2G,CDKN2B,brain microvascular endothelial cell from ENCDO227AAA,0.461345,chr9,22103218,22104128
180,ENCODE-rE2G,CDKN2B,fibroblast of pulmonary artery from ENCDO245AAA,0.386076,chr9,22103209,22104129
296,ENCODE-rE2G,CDKN2B,fibroblast of the aortic adventitia from ENCDO175AAA,0.338397,chr9,22103216,22104126
187,ENCODE-rE2G,CDKN2B,"cardiovascular progenitor cell from ENCDO221AAA treated with Activin A, Bone morphogenetic prote...",0.337571,chr9,22103210,22103710
593,scE2G,CDKN2B,coronary artery,0.232632,chr9,22103172,22104040


... (9 outputs left out)

== adastra ==


,tf,biological_context,favored,p_adj
0,RELA,"HUVEC-C (HUVEC, umbilical vein endothelial cells)",alt,0.048881
1,HIF1A,EA.hy 926 (human umbilical vein endothelial cell line),alt,0.002893


== e2g_in_context ==


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end
1623,ENCODE-rE2G,NOS3,kidney capillary endothelial cell from ENCDO987XTQ,0.999996,chr7,150992745,150994970
1619,ENCODE-rE2G,NOS3,ascending aorta from ENCDO793LXB,0.999994,chr7,150992699,150994053
1756,scE2G,NOS3,endothelial cell of lymphatic vessel,0.999946,chr7,150992814,150993314
1754,scE2G,NOS3,endothelial cell,0.996977,chr7,150992720,150994053
1261,ENCODE-rE2G,NOS3,dermis blood vessel endothelial cell from ENCDO244AAA,0.969839,chr7,150992684,150993184
1315,ENCODE-rE2G,NOS3,dermis microvascular lymphatic vessel endothelial cell from ENCDO242AAA,0.965207,chr7,150992688,150993188
1075,ENCODE-rE2G,NOS3,endothelial cell from ENCDO336AAA,0.948126,chr7,150992674,150993174
1148,ENCODE-rE2G,NOS3,endothelial cell of umbilical vein from ENCDO560UMK,0.947788,chr7,150992678,150993178
1182,ENCODE-rE2G,NOS3,lung microvascular endothelial cell from ENCDO238AAA,0.946232,chr7,150992680,150993180
1273,ENCODE-rE2G,NOS3,lung microvascular endothelial cell from ENCDO237AAA,0.942506,chr7,150992685,150993185


- rs1537373: myocardial infarction 44.4, coronary artery calcification 10.4, brain aneurysm 28.5; scE2G coronary artery → CDKN2A 0.32 and CDKN2B 0.23; ENCODE-rE2G in aortic adventitial fibroblasts and microvascular endothelium; HepG2 MPRA +1.03, which is not a vascular cell type.
- rs3918226: hypertension 62.1, cardiovascular disorder 48.7; endothelial E2G → NOS3 in 34 contexts, up to 1.0.
- rs2070744 (T-786C): ADASTRA HIF1A in EA.hy926 (adjusted p 0.003) and RELA in HUVEC (0.049).

In [23]:
#check LD blocks
p21_ld = ld_block("NC_000009.12:22103341:T:G", "CDKN2B", SMC_CONTEXT)
print(len(p21_ld), "variants in the block (lead + proxies)")
p21_ld

58 variants in the block (lead + proxies)


,variant,rsid,r2,gwas,screen_hits,reporter_hits,cv2f_max,e2g_n_contexts,e2g_max_score,caqtl,adastra
0,NC_000009.12:22103341:T:G,[rs1537373],1.000,myocardial infarction 44.4; brain aneurysm 28.5; coronary artery calcification 10.4,[],[],None,6,0.63701,[],[]
1,NC_000009.12:22090936:T:G,[rs9644862],0.993,vascular disorder 13.5; carotid plaque build 11.0,[],[],None,0,NaN,[],[]
2,NC_000009.12:22090935:C:T,[rs9644861],0.993,cataract 14.7; diastolic blood pressure 7.3,[],[],None,0,NaN,[],[]
3,NC_000009.12:22103813:A:G,[rs1333042],0.992,coronary artery disorder 71.7; parental longevity 14.2,[],[],None,5,0.63701,[],[]
4,NC_000009.12:22100176:G:C,[rs1556516],0.990,coronary artery disorder 53.1; parental longevity 19.9; father's age at death 14.7,[],[],None,0,NaN,[],[]
5,NC_000009.12:22092257:G:A,[rs2210538],0.990,,[],[],None,0,NaN,[],[]
6,NC_000009.12:22091924:G:T,[rs7866503],0.990,abdominal aortic aneurysm 12.7; brain aneurysm 12.7; thoracic aortic aneurysm 12.7,[],[],None,0,NaN,[],[]
7,NC_000009.12:22099568:C:A,[rs1537371],0.989,angina pectoris 100.3; coronary atherosclerosis 86.6; coronary artery bypass 85.7,[],[],None,0,NaN,[],[]
8,NC_000009.12:22090521:G:A,[rs9644859],0.988,,[],[],None,0,NaN,[],[]
9,NC_000009.12:22090603:C:T,[rs9644860],0.988,familial hypercholesterolemia 11.4,[],[],None,0,NaN,[],[]


A 58-variant block (57 proxies) that includes the lead CAD variants rs2891168 (CAD 294) and rs4977574 (CAD 222), and rs1333042 (r² 0.99, CAD 71.7). ADASTRA hits in the block are from prostate lines.

In [24]:
for rs, alt in [("rs2070744", "T"), ("rs3918226", None)]:     # rs2070744 has two alt alleles; ADASTRA data are on C>T
    vid = variant_id_for(rs, alt)
    print(f"== {rs}  {vid} ==")
    proxies = ld(vid)
    proxies["rsid"] = proxies["variant"].map(lambda v: variant_node(v).get("rsid"))
    display(proxies)

== rs2070744  NC_000007.14:150992990:C:T ==


,variant,r2,rsid
0,NC_000007.14:150992854:G:A,0.966,[rs1800779]
1,NC_000007.14:150992308:A:T,0.930,[rs1800783]
2,NC_000007.14:150987459:A:G,0.921,[rs6946415]
3,NC_000007.14:150983888:G:C,0.919,[rs2373962]
4,NC_000007.14:150983417:T:G,0.918,[rs4496877]
5,NC_000007.14:150984825:T:C,0.917,[rs6951150]
6,NC_000007.14:150987257:A:G,0.917,[rs6946091]
7,NC_000007.14:150981954:C:G,0.912,[rs6953274]
8,NC_000007.14:150988710:A:,0.904,[rs5888424]
9,NC_000007.14:150978779:A:G,0.891,[rs11771808]


== rs3918226  NC_000007.14:150993087:C:T ==


,variant,r2,rsid


rs2070744 ↔ rs1800779 at r² 0.97; rs3918226 has no EUR proxies.

In [25]:
# Do rs1537373 and rs1333042 share a predicted vascular element?
VASC = SMC_CONTEXT + "|endothel|fibroblast"
a = in_context(e2g("NC_000009.12:22103341:T:G"), VASC).assign(rsid="rs1537373")
b = in_context(e2g(variant_id_for("rs1333042")), VASC).assign(rsid="rs1333042")
both = pd.concat([a, b], ignore_index=True)
el = ["genomic_element.chr", "genomic_element.start", "genomic_element.end"]
shared = both.groupby(el + ["method", "biological_context"])["rsid"].agg(lambda s: sorted(set(s))).reset_index()
display(shared[shared["rsid"].map(len) == 2])                       # elements that contain both variants
display(both[both["gene_name"].isin(["CDKN2A", "CDKN2B"])].sort_values("score", ascending=False).head(20))

rs1333042 maps to ['NC_000009.12:22103813:A:C', 'NC_000009.12:22103813:A:G']; using the first


,genomic_element.chr,genomic_element.start,genomic_element.end,method,biological_context,rsid
0,chr9,22103159,22104067,scE2G,fibroblast,"[rs1333042, rs1537373]"
2,chr9,22103172,22104040,scE2G,coronary artery,"[rs1333042, rs1537373]"
3,chr9,22103178,22104093,ENCODE-rE2G,fibroblast of dermis from ENCDO267AAA,"[rs1333042, rs1537373]"
4,chr9,22103193,22104088,ENCODE-rE2G,foreskin fibroblast from ENCDO191CQJ,"[rs1333042, rs1537373]"
5,chr9,22103202,22104124,ENCODE-rE2G,cardiac fibroblast from ENCDO095AAA,"[rs1333042, rs1537373]"
6,chr9,22103204,22104098,ENCODE-rE2G,fibroblast of the conjunctiva from ENCDO231AAA,"[rs1333042, rs1537373]"
8,chr9,22103207,22104096,ENCODE-rE2G,fibroblast of gingiva from ENCDO235AAA,"[rs1333042, rs1537373]"
9,chr9,22103209,22104094,ENCODE-rE2G,cardiac fibroblast from ENCDO230AAA,"[rs1333042, rs1537373]"
10,chr9,22103209,22104129,ENCODE-rE2G,fibroblast of pulmonary artery from ENCDO245AAA,"[rs1333042, rs1537373]"
12,chr9,22103210,22104108,ENCODE-rE2G,foreskin fibroblast from ENCDO232AAA,"[rs1333042, rs1537373]"


,method,gene_name,biological_context,score,genomic_element.chr,genomic_element.start,genomic_element.end,rsid
4,ENCODE-rE2G,CDKN2A,cardiac fibroblast from ENCDO754GEB,0.917635,chr9,22103227,22104076,rs1537373
136,ENCODE-rE2G,CDKN2A,cardiac fibroblast from ENCDO754GEB,0.917635,chr9,22103227,22104076,rs1333042
140,ENCODE-rE2G,CDKN2A,fibroblast of skin of scalp from ENCDO576LXR,0.881981,chr9,22103211,22104088,rs1333042
8,ENCODE-rE2G,CDKN2A,fibroblast of skin of scalp from ENCDO576LXR,0.881981,chr9,22103211,22104088,rs1537373
10,ENCODE-rE2G,CDKN2A,fibroblast of skin of scalp from ENCDO576LXR,0.872840,chr9,22103213,22104071,rs1537373
142,ENCODE-rE2G,CDKN2A,fibroblast of skin of scalp from ENCDO576LXR,0.872840,chr9,22103213,22104071,rs1333042
18,ENCODE-rE2G,CDKN2A,fibroblast of skin of back from ENCDO597UXW,0.837926,chr9,22103217,22104089,rs1537373
150,ENCODE-rE2G,CDKN2A,fibroblast of skin of back from ENCDO597UXW,0.837926,chr9,22103217,22104089,rs1333042
19,ENCODE-rE2G,CDKN2B,fibroblast of lung from ENCDO269AAA,0.832362,chr9,22103213,22104123,rs1537373
151,ENCODE-rE2G,CDKN2B,fibroblast of lung from ENCDO269AAA,0.832362,chr9,22103213,22104123,rs1333042


at least one vascular E2G element contains both variants, linked to CDKN2A and CDKN2B.

Note
- Page-size limits by endpoint (from the router code): `/variants/phenotypes`, `/variants/biosamples`, `/variants/proteins`, `/phenotypes/variants` 100; `/variants/genomic-elements` 300; `/variants/genes`, `/variants/variant-ld`, `/genomic-elements/genes`, `/files-filesets` 500; `/genomic-elements` 1000.

### Evidence summary

| Locus | Evidence | Value |
| :-- | :-- | :-- |
| 9p21 | element perturbed in HCASMC (TAP-seq CRISPRi) | chr9:22103301-22103601 |
| 9p21 | effect on genes | CDKN2B −0.17 (adjusted p 1.1e-5); CDKN2A −0.09 (8.3e-5); MTAP and CDKN2B-AS1 not significant |
| 9p21 | variant in the element | rs1537373 |
| 9p21 | GWAS | myocardial infarction 44.4; coronary artery calcification 10.4 |
| 9p21 | TopLD EUR, r² ≥ 0.8 | 57 proxies, including rs2891168 (CAD 294) and rs4977574 (CAD 222) |
| 9p21 | element to gene, vascular | scE2G coronary artery: CDKN2A 0.32, CDKN2B 0.23 |
| NOS3 | element perturbed in TeloHAEC (CRISPRi FlowFISH) | chr7:150992690-150993190 |
| NOS3 | effect on gene | NOS3 −2.2 to −2.4 in four screens (adjusted p down to 5e-28) |
| NOS3 | rs3918226, GWAS | hypertension 62.1; cardiovascular disorder 48.7 |
| NOS3 | rs2070744, ADASTRA | HIF1A in EA.hy926 (adjusted p 0.003); RELA in HUVEC (0.049) |
| NOS3 | element to gene, endothelial | rE2G / scE2G to NOS3, up to 1.0 |

### Summary
Starting from CRISPR perturbation screens in the two vascular cell types where coronary artery disease risk acts, the catalog connects risk variants to the genes their elements control. In coronary artery smooth muscle cells, silencing an element in the 9p21 locus lowers CDKN2B and CDKN2A but not MTAP or CDKN2B-AS1; the element contains rs1537373, which is associated with myocardial infarction and sits in a 58-variant LD block with the lead CAD variants. In aortic endothelial cells, silencing the NOS3 promoter element reduces NOS3 in every replicate screen; the element contains rs3918226, associated with hypertension, and rs2070744, which shows allele-specific HIF1A and RELA binding in endothelial cell lines. Neither locus has a reporter assay in a vascular cell type, so the variant-level effect is not yet tested directly.